<img src="https://github.com/NVIDIA/accelerated-computing-hub/blob/main/gpu-cpp-tutorial/notebooks/01.03-Extending-Algorithms/Images/nvidia_header.png?raw=1" style="margin-left: -30px; width: 300px; float: left;">

## Exercise: Computing Variance

So far, you've learned how to compute mean variance and maximal difference.
In this exercise, you'll apply techniques of extending standard algorithms that we just covered.
This time, you'll be implementing an efficient variance algorithm.
Variance is computed on a sequence of data.
It measures how far the values in the sequence are spread from the mean:

$$
\frac{\sum\left(x_{i} - \overline{x} \right)^{2}}{N}
$$

As the equation above suggests, for each value in the sequence we have to compute the squared difference between this value and mean.
We then add all those squared differences together and divide the resulting sum by `N`.

The next exercise consists of using a transform iterator to compute the squared differences.

Transform iterator API for your reference:

```c++
int constant = 2;
auto transform_it = thrust::make_transform_iterator(
    // iterator to the beginning of the input sequence
    vector.begin(),
    // capture constant in the lambda by value with `[name]`
    [constant]__host__ __device__(float value_from_input_sequence) {
      // transformation of each element
      return value_from_input_sequence * constant;
    });
```

Use `thrust::reduce` to compute the sum of squared differences.

In [4]:
#@title Google Colab Setup
!mkdir -p Sources
!wget https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/gpu-cpp-tutorial/notebooks/01.03-Extending-Algorithms/Sources/ach.h -nv -O Sources/ach.h
!sudo apt-key adv --fetch-keys https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64/7fa2af80.pub > /dev/null 2>&1
!sudo add-apt-repository -y "deb https://developer.download.nvidia.com/devtools/repos/ubuntu$(source /etc/lsb-release; echo "$DISTRIB_RELEASE" | tr -d .)/$(dpkg --print-architecture)/ /" > /dev/null 2>&1
!sudo apt install -y nsight-systems > /dev/null 2>&1

2025-04-01 03:49:55 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/gpu-cpp-tutorial/notebooks/01.03-Extending-Algorithms/Sources/ach.h [2787/2787] -> "Sources/ach.h" [1]
2025-04-01 03:49:56 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/gpu-cpp-tutorial/notebooks/02.02-Asynchrony/Sources/nvtx3.hpp [105838/105838] -> "Sources/nvtx3.hpp" [1]


In [25]:
%%writefile Sources/variance.cpp
#include "ach.h"
#include <cstdlib>
#include "thrust/random.h"
#include <thrust/device_vector.h>

float variance(const thrust::universal_vector<float> &x, float mean) {
  // update the following line so that dereferencing `squared_differences`
  // returns `(xi - mean) * (xi - mean)`
  auto squared_differences = thrust::make_transform_iterator(
    x.begin(), [mean] __host__ __device__(float value) {
      return (value - mean) * (value - mean);
    });

  return thrust::reduce(thrust::device, squared_differences,
                        squared_differences + x.size()) /
         x.size();
}

// Compute the mean
float mean(thrust::universal_vector<float> vec) {
  return thrust::reduce(thrust::device, vec.begin(), vec.end()) / vec.size();
}

int main() {
  float ambient_temp = 20;
  // thrust::universal_vector<float> prev{42, 24, 50};
  // thrust::universal_vector<float> next{0, 0, 0};

  // allocate vectors containing 2^28 elements
  thrust::device_vector<float> prev(1 << 28);
  thrust::device_vector<float> next(1 << 28);

  // thrust::sequence(prev.begin(), prev.end());
  // thrust::sequence(next.rbegin(), next.rend());

  // srand(13);
  thrust::minstd_rand rng1;
  thrust::generate(thrust::device, prev.begin(), prev.end(), rng1);

  std::printf("step  variance\n");
  for (int step = 0; step < 3; step++) {
    thrust::transform(thrust::device, prev.begin(), prev.end(), next.begin(),
                      [=] __host__ __device__(float temp) {
                        return temp + 0.5 * (ambient_temp - temp);
                      });
    std::printf("%d     %.2f\n", step, variance(next, mean(next)));
    {
      nvtx3::scoped_range r2{std::string("Swapping data")};
      next.swap(prev);
    }
  }
}

Overwriting Sources/variance.cpp


In [26]:
!nvcc --extended-lambda -o /tmp/a.out Sources/variance.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

step  variance
0     2082966259892224.00
1     520741632081920.00
2     130185391243264.00


In [27]:
!nsys profile --force-overwrite true -o variance_device_vector /tmp/a.out # run and profile executable

step  variance
0     2082966259892224.00
1     520741632081920.00
2     130185391243264.00
Generating '/tmp/nsys-report-2abc.qdstrm'
[1/1] [========================100%] variance_device_vector.nsys-rep
Generated:
	/content/variance_device_vector.nsys-rep


In [28]:
!ncu --kernel-name static_kernel --launch-skip 0 --launch-count 1 "/tmp/a.out"

==PROF== Connected to process 8656 (/tmp/a.out)
==PROF== Profiling "static_kernel": 0%....50%....100% - 9 passes
step  variance
0     2082966259892224.00
1     520741632081920.00
2     130185391243264.00
==PROF== Disconnected from process 8656
[8656] a.out@127.0.0.1
  void static_kernel<policy_350_t, unsigned long, functor<device_ptr<float>, float>>(T2, T3) (524288, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.99
    SM Frequency                    Mhz       585.00
    Elapsed Cycles                cycle    2,715,670
    Memory Throughput                 %        84.19
    DRAM Throughput                   %        84.19
    Duration                         ms         4.64
    L1/TEX Cache Throughput           %        61.85
 

The output of your program should be:

| Step | Variance |
|------|----------|
| 0    | 29.56    |
| 1    | 7.39     |
| 2    | 1.85     |

If you’re unsure how to proceed, consider expanding this section for guidance. Use the hint only after giving the problem a genuine attempt.

<details>
  <summary>Hints</summary>
  
  - You can capture mean in a lambda function with `[mean]__host__ __device__ (...`
  - You can transform the input sequence into squared differences with `thrust::transform_iterator`
  - You can create a transform iterator with `thrust::make_transform_iterator`
</details>

Open this section only after you’ve made a serious attempt at solving the problem. Once you’ve completed your solution, compare it with the reference provided here to evaluate your approach and identify any potential improvements.

<details>
  <summary>Solution</summary>

  Key points:
  - use `thrust::make_transform_iterator`
  - capture mean by value in lambda

  Solution:
  ```c++
  auto squared_differences = thrust::make_transform_iterator(
    x.begin(), [mean] __host__ __device__(float value) {
      return (value - mean) * (value - mean);
    });
  ```

  You can find full solution [here](Solutions/variance.cu).
</details>

---
Congratulations!  Now that you know how to extend standard algorithms, proceed to the [next section](../01.04-Vocabulary-Types/01.04.01-Vocabulary-Types.ipynb).

<img src="https://github.com/NVIDIA/accelerated-computing-hub/blob/main/gpu-cpp-tutorial/notebooks/01.03-Extending-Algorithms/Images/nvidia_header.png?raw=1" style="margin-left: -30px; width: 300px; float: left;">